In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
import batman
from transitleastsquares import transitleastsquares
import seaborn as sns
import os
from functools import lru_cache
from itertools import product
from scipy.signal import savgol_filter

# ------------------------------------
# 1. Cached light curve loader
# ------------------------------------
@lru_cache(maxsize=10)
def get_lightcurve(tic):
    lc_collection = lk.search_lightcurve(tic, mission="TESS", author="TESS-SPOC", cadence="long").download_all(quality_bitmask="hard")
    if lc_collection is None or len(lc_collection) == 0:
        return None
    
    lc = lc_collection.stitch().remove_nans()



    #plt.plot(lc.time.value, lc.flux.value, 'k.', markersize=1, alpha=0.5)


    return lc.to_pandas().reset_index()

# ------------------------------------
# 2. Flare masking: ±3hr from >5σ events
# ------------------------------------
def mask_flares(df):
    median = df['flux'].median()
    std = df['flux'].std()
    threshold = median + 5 * std
    flare_times = df.loc[df['flux'] > threshold, 'time'].values

    def within_3hr(t):
        return np.any(np.abs(flare_times - t) <= 0.125)

    flare_mask = df['time'].apply(within_3hr)
    return df[~flare_mask].reset_index(drop=True)

# ------------------------------------
# 3. Inject transit only (returns multiplier)
# ------------------------------------
def inject_model(time, period, rp_earth, st_radius, st_mass):
    # Constants
    R_earth = 6.371e6        # meters
    R_sun = 6.957e8          # meters
    M_sun = 1.989e30         # kg
    G = 6.67430e-11          # m^3 / kg / s^2

    # Convert radius: Earth radii to stellar radii
    rp_rstar = (rp_earth * R_earth) / (st_radius * R_sun)

    # Convert mass to kg and period to seconds
    period_sec = period * 86400
    st_mass_kg = st_mass * M_sun
    st_radius_m = st_radius * R_sun

    # Compute semi-major axis in meters
    a_meters = ((G * st_mass_kg * period_sec**2) / (4 * np.pi**2))**(1 / 3)

    # Convert semi-major axis to stellar radii
    a_rstar = a_meters / st_radius_m

    print(f"rp/rstar^2: {rp_rstar**2:.6f}, a/rstar: {a_rstar:.2f}, period: {period:.2f} days")

    # Set up transit parameters
    mparams = batman.TransitParams()
    mparams.t0 = np.random.uniform(0, period)
    mparams.per = period
    mparams.rp = rp_rstar
    mparams.a = a_rstar
    mparams.inc = 90
    mparams.ecc = 0
    mparams.w = 90
    mparams.u = [0.0, 0.0]
    mparams.limb_dark = "quadratic"

    model = batman.TransitModel(mparams, time)
    #plt.plot(time, model.light_curve(mparams), color='red', alpha=0.5, label='Injected Transit')
    #plt.xlim(1650,1700)

    #plt.show()


    #return model.light_curve(mparams)
    return np.ones_like(time)


# 4. Run TLS detection for a single injection
# ------------------------------------
def run_tls(tic, time, flux, true_period, plots=False):
    # cadence_days = time[1] - time[0]
    # duration_days = 2
    # wl = int(np.round(duration_days / cadence_days))
    # if wl % 2 == 0:
    #     wl += 1  # make it odd

    # print(wl)
    # flux_filter = savgol_filter(flux, wl, 2)
    # flux = flux / flux_filter

    ##LC PLOT
    #plt.plot(time, flux, 'k.', markersize=1, alpha=0.5)
    #plt.title(f"Injected Lightcurve for {tic} flattened")
    #plt.xlim(1650,1700)
    #plt.show()

    if len(time) == 0 or len(flux) == 0:
        return False
    tls = transitleastsquares(time, flux)
    result = tls.power(period_max = 25)

    powers = result.power
    periods = result.periods

    periodogram = pd.DataFrame({
        'period': periods,
        'power': powers
    })

    power_7_periods = periodogram[periodogram['power'] > 7]['period'].values
    print(f"TLS Periods with power > 7: {power_7_periods}")
    power_7_power = periodogram[periodogram['power'] > 7]['power'].values


    print(f"{tic}, FAP: {result.FAP}, SDE: {result.SDE}, True Period: {true_period}")
    # Mark as found if period matches true period or a simple alias (1/2x, 2x, 1/3x, 3x, etc.)
    aliases = [true_period, true_period / 2, true_period * 2, true_period / 3, true_period * 3]
    #found = result.FAP < 1e-4 and any(0.99 * alias < result.period < 1.01 * alias for alias in aliases)
    found = any(0.99 * alias < hp < 1.01 * alias for hp in power_7_periods for alias in aliases)


    alias_found = any(0.99 * alias < result.period < 1.01 * alias for alias in aliases)
    print(f"Detection found: {found}")
    if alias_found == True and found == False:
        print(f"True transit or alias found, but high FAP: {alias_found}")
    #print("TLS Transit duration (hours):", result.duration * 24)

    if plots == True:
        #TLS phase fold
        plt.figure()
        plt.scatter(result.folded_phase, result.folded_y, color='blue', s=10, alpha=0.5,)
        plt.plot(result.model_folded_phase, result.model_folded_model, color='red')
        plt.title(f"TLS Phase Folded Lightcurve for {tic}")
        plt.xlabel('Phase')
        plt.ylabel('Relative flux');
        plt.show()

        #Periodogram
        plt.figure()
        ax = plt.gca()
        ax.axvline(result.period, alpha=0.4, lw=3)
        plt.xlim(np.min(result.periods), np.max(result.periods))
        for n in range(2, 10):
            ax.axvline(n*result.period, alpha=0.4, lw=1, linestyle="dashed")
            ax.axvline(result.period / n, alpha=0.4, lw=1, linestyle="dashed")
        plt.ylabel(r'SDE')
        plt.xlabel('Period (days)')
        plt.plot(result.periods, result.power, color='black', lw=0.5)
        plt.xlim(0, max(result.periods));
        #plt.xlim(20.8, 21.0)
        plt.show()


    

    return found, alias_found, result.period, result.SDE, power_7_periods, power_7_power, result.period_uncertainty, result.depth, result.rp_rs, result.FAP, result.snr


# ------------------------------------
# 5. Run detection across grid 
# ------------------------------------
def run_detection_for_tic(tic, periods, radii, save_file, plots=False, xlim=None):
    
    lc = get_lightcurve(tic)
    if lc is None:
        print(f"❌ No light curve for {tic}")
        return pd.DataFrame()

    base_df = lc[['time', 'flux']].copy()
    

    raw_time = base_df['time'].values
    raw_flux = base_df['flux'].values
    if plots == True:
        plt.plot(raw_time, raw_flux, 'k.', markersize=1, alpha=0.5)
        plt.title(f"Bare Lightcurve for {tic}")
        if xlim is not None:
            plt.xlim(xlim)
        plt.show()
    
   

    # Get stellar parameters
    tic_id = int(tic.split()[1])
    star_params = pd.read_csv("stellar_params_CTL.csv").query(f"id == {tic_id}")
    if star_params.empty:
        print(f"⚠️ No stellar params for {tic}")
        return pd.DataFrame()

    st_radius = star_params['RAD'].values[0]
    st_mass = star_params['MASS'].values[0] 
    st_teff = star_params['teff'].values[0]
    st_tmag = star_params['tmag'].values[0]

    results = pd.DataFrame()
     # Loop over period-radius grid
    for period, rp_earth in product(periods, radii):
        detections = 0
        for trial in range(trials):
            print(f"Running trial {trial + 1}/{trials} for {tic}, Period: {period}, Rp (Earth Radii): {rp_earth}")
            # plt.plot(raw_time, raw_flux, 'k.', markersize=1, alpha=0.5)
            # plt.title(f"Bare Lightcurve for {tic}")
            # #plt.xlim(1650,1700)
            # plt.show()
            transit_signal = inject_model(raw_time, period, rp_earth, st_radius, st_mass)
            injected_flux = raw_flux * transit_signal
            injected_flux_mean_normalized = injected_flux #/ np.mean(injected_flux)
            injected_df_mean_normalized = pd.DataFrame({'time': raw_time, 'flux': injected_flux_mean_normalized})
            flux_normalized_flares_removed = mask_flares(injected_df_mean_normalized)

            if plots == True:
                plt.plot(flux_normalized_flares_removed['time'], flux_normalized_flares_removed['flux'], 'k.', markersize=1, alpha=0.5)
                plt.title(f"Injected Lightcurve for tic {tic} unflattened")
                if xlim is not None:
                    plt.xlim(xlim)
                plt.show()

            lc = lk.LightCurve(time=flux_normalized_flares_removed['time'], flux=flux_normalized_flares_removed['flux'])
            time = lc.time.value  # in days
            if len(time) < 2:
                return None

            cadence_days = time[1] - time[0]
            duration_days = 2
            wl = int(np.round(duration_days / cadence_days))
            if wl % 2 == 0:
                wl += 1  # make it odd, as required

            print(f"Cadence: {cadence_days:.5f} days, window_length: {wl}")

            lc_flat = lc.flatten(window_length=wl, polyorder=2)
            flat_df = lc_flat.to_pandas().reset_index()
            time = flat_df['time'].values
            flux = flat_df['flux'].values
            #flux = flux / np.median(flux)
            if plots == True:
                plt.plot(time, flux, 'k.', markersize=1, alpha=0.5)
                plt.title(f"Injected Lightcurve for {tic} flattened")
                if xlim is not None:
                    plt.xlim(xlim)
                plt.show()

            true_period = period
        
            found, alias_found, tls_period, SDE_strongest,  power_array_7, SDE_array_7, tls_period_uncertainty, tls_depth, tls_rp_rs, tls_FAP, tls_snr = run_tls(tic,time, flux, true_period, plots=plots)

            detections += found

            row = {
                'TIC': tic,
                'True Radius (Earth Radii)': round(float(rp_earth), 2),
                'True Period (Days)': round(float(period), 4),
                'Stellar Radius': round(float(st_radius), 2),
                'Stellar Temperature': st_teff,
                'Stellar Magnitude': st_tmag,
                'Detection': found,
                'TLS Period': tls_period,
                'TLS SDE strongest': SDE_strongest,
                'TLS SDE > 7 array': SDE_array_7,
                'TLS Periods array': power_array_7,
                'TLS Period Uncertainty': tls_period_uncertainty,
                'TLS Depth': tls_depth,
                'TLS Rp/Rs': tls_rp_rs,
                'TLS FAP': tls_FAP,
                'TLS SNR': tls_snr,
                #'Detection %': detections / trials,
                '# trials': trial
            }


            # Convert to DataFrame and append to file

            row_df = pd.DataFrame([row])

            if os.path.exists(save_file):
                df = pd.read_csv(save_file)
                unique_cols=["TIC", "True Period (Days)", "True Radius (Earth Radii)", "# trials"]
                # Drop any row that matches all values in unique_cols
                condition = (df[unique_cols[0]] == row[unique_cols[0]])
                for col in unique_cols[1:]:
                    condition &= (df[col] == row[col])
                df = df[~condition]

                # Append new row and save
                df = pd.concat([df, row_df], ignore_index=True)
            else:
                df = row_df

            df.to_csv(save_file, index=False)

            # #append simply
            # pd.DataFrame([row]).to_csv(save_file, mode='a', header=not os.path.exists(save_file), index=False)
            results = pd.concat([results, row_df], ignore_index=True)

    return results


# ------------------------------------
# 6. Save heatmap and results
# ------------------------------------
def save_heatmap(data, tic):
    data = data[['True Radius (Earth Radii)', 'True Period (Days)', 'Detection %']]
    pivot = data.pivot(index="True Radius (Earth Radii)", columns="True Period (Days)", values="Detection %")
    plt.figure(figsize=(8, 6))
    sns.heatmap(pivot, cmap="viridis", vmin=0, vmax=1, cbar_kws={'label': 'Percent found'}).invert_yaxis()
    plt.title(f"{tic} noise transit detection rate")
    plt.xlabel("Period (Days)")
    plt.ylabel("Radius (Earth Radii)")
    plt.tight_layout()
    os.makedirs("heatmap_data", exist_ok=True)
    plt.savefig(f'heatmap_data/{tic.replace(" ", "_")}_cel_heatmap.png', dpi=300)
    

    path = f'heatmap_data/{tic.replace(" ", "_")}_cel_heatmap.csv'
    data.to_csv(path, index=False)
    print(f"✅ Saved: {path}")
    plt.show()
    


# ------------------------------------
# 7. Main driver
# ------------------------------------
if __name__ == "__main__":

    #save_file = "./inj_results/all_results_to_25P_multi_savgolay_2dayfilter_overSDE7.csv"
    os.makedirs("inj_rec_results", exist_ok=True)

    save_file = './inj_rec_results/test_min_periods' #rec_only_weird_TIC_recovery.csv'

    # Load TIC IDs from the CSV file
    star_params = pd.read_csv("stellar_params_CTL.csv")
    #print(star_params.head(20))

    #star_params = star_params.iloc[:1]  # Limit to first 10 for testing, remove this line for full run
    #star_params = star_params.iloc[11:12] 


    ###WEIRD TIC IDS - also change batman which runs and period/radius blocks


    #star_ids = [241204178, 103112722, 1102352744, 114827044,114859567,120566166,121048612,124533795,385919695,387628382,389296668,394026461,17905814,257507933,258060707,25978649,26412365,264722728,26647984,266500596,269829146,270301360,469870550,47402214,52211797,313403564,313988572,314713177,315104364,316096209,328466136,330123048,178284735]
    #star_params = star_params[star_params['id'].isin(star_ids)]
    #for each pair of toi and tic, there's a true period and rp_earth to run

    df_save = pd.DataFrame()

    
for _, row in star_params.iterrows():
    tic = f"TIC {int(row['id'])}"
    print(tic)
    periods = np.exp(np.linspace(-.52, 5.77, num=26))[1:13][::2]
    radii = np.exp(np.linspace(1.63875, 0.36125, num=14))
    #radii = radii[:1] 

    # Log-spaced list between 1.4 and 5 (inclusive), in log10 space
    radii = np.logspace(np.log10(1.4), np.log10(5), num=6)
    # Log-spaced list between 0.7 and 25 (inclusive), in log10 space
    periods = np.logspace(np.log10(0.7), np.log10(25), num=7)
    radii = [10]
    periods = [.53]
    print(periods, radii)
    #radii = [1,2,3,4,5]
    #periods = [1,3,5,7]
    #periods = periods[:2]
    #radii = radii[-2:]
    trials = 1

    try:
        df_results = run_detection_for_tic(tic, periods, radii, save_file, plots=False, xlim=None)
        save_heatmap(df_results, tic)

    except Exception as e:
        print(f"❌ Error occurred for {tic}: {e}")
        df_results = pd.DataFrame([{
            'TIC': tic,
            'Stellar Radius': None,
            'Stellar Temperature': None,
            'Stellar Magnitude': None,
            'True Radius (Earth Radii)': None,
            'True Period (Days)': None,
            'Detection': e,
            'TLS Period': None,
            'TLS SDE strongest': None,
            'TLS SDE > 7 array': None,
            'TLS Periods array': None,
            'TLS Period Uncertainty': None,
            'TLS Depth': None,
            'TLS Rp/Rs': None,
            'TLS FAP': None,
            'TLS SNR': None,
            #'Detection %': None,
            '# trials': None
        }])


        updated = df_results
        updated.to_csv(save_file, mode='a', index=False, header=False)

        #save_heatmap(df_results, tic)
#run for all M dwarfs

#implement save_heatmap and inject_model

TIC 158235404
[0.53] [10]
Running trial 1/1 for TIC 158235404, Period: 0.53, Rp (Earth Radii): 10
rp/rstar^2: 0.258837, a/rstar: 11.91, period: 0.53 days
Cadence: 0.02083 days, window_length: 97
Transit Least Squares TLS 1.0.31 (22 Nov 2021)
Creating model cache for 42 durations
Searching 4486 data points, 40668 periods from 0.602 to 24.996 days
Using all 10 CPU threads


100%|██████████| 40668/40668 periods | 00:35<00:004


Searching for best T0 for period 0.64869 days


/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/numpy/core/_methods.py:264: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/numpy/core/_methods.py:222: RuntimeWarning: invalid value encountered in true_divide
  arrmean = um.true_divide(arrmean, div, out=arrmean, casting='unsafe',
/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/numpy/core/_methods.py:256: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/transitleastsquares/main.py:411: UserWarning: 616 of 626 transits without data. The true period may be twice the given period.
  warnings.warn(text)


TLS Periods with power > 7: [0.64809928 0.648169   0.64827361 0.64834336 0.64841312 0.64844801
 0.64851778 0.64858757 0.64862247 0.64869227 0.64876208 0.64879699
 0.64886681 0.64893665 0.69795757 0.69803453 0.698227   0.69830401
 0.69834252 0.69841954 0.69849658 0.69861216 0.69868922 0.6987663
 0.69888193 0.74780807 0.74789245 0.74797685 0.74801905 0.74810347
 0.74818789 0.74831456 0.74839902 0.74848349 0.74856798 0.74861023
 0.74869473 0.79744379 0.79776562 0.7978576  0.7979496  0.79808762
 0.79817965 0.7982717  0.79840979 0.79850188 0.79859397 0.84752045
 0.84762016 0.84771988 0.84796927 0.84806905 0.84816885 0.84826866
 0.84841841 0.84851826 0.89723754 0.89761417 0.89772181 0.89782947
 0.89793715 0.89831417 0.89842193 0.9472105  0.94732615 0.94738398
 0.94744182 0.94749966 0.94761535 0.94773107 0.9478468  0.94796255
 0.94807832 0.99748743 0.99761134 0.99773526 0.99785921 0.99798318
 0.99810716 0.99823117 1.04721121 1.04747564 1.04774017 1.14689661
 1.19695642 1.19711442 1.19727245 1

...

TLS Periods with power > 7: [ 2.71923201  3.50809214  5.13815333  5.29645612  6.23040104  6.25777963
  6.25897364  6.38360141  7.02408117  7.40645039  9.08785476  9.09571286
 11.27989053 11.42783386 11.47861647 11.58108793 11.58380098 11.58651489
 11.85944246 12.47040081 12.47339516 12.47639047 12.48538214 12.48838129
 12.49438246 12.50339143 12.52143537 12.52745773 12.53047036 12.53348395
 12.83978933 12.84290252 12.84601672 12.84913193 12.88659311 13.26934037
 13.50295904 13.70128261 13.70467739 16.03271732 16.03690343 16.04109099
 17.65329478 17.65805439 19.03465296 19.25201925 19.25736212 19.26270696
 19.26805378 19.27340258 19.27875336 21.17663318 21.18269986 21.18876886
 21.19484017 21.2009138  21.20698976 22.95648216 23.16715433 23.17399304
 23.55430487 23.56129639 24.67187114 24.94150828]
TIC 1102352744, FAP: 8.0032e-05, SDE: 43.43445842394561, True Period: 0.53
Detection found: False
❌ Error occurred for TIC 1102352744: "['Detection %'] not in index"
TIC 213841827
[0.53] [10]


100%|██████████| 76411/76411 periods | 2:00:45<00:00


Searching for best T0 for period 1.89374 days


/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/numpy/core/_methods.py:264: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/numpy/core/_methods.py:222: RuntimeWarning: invalid value encountered in true_divide
  arrmean = um.true_divide(arrmean, div, out=arrmean, casting='unsafe',
/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/numpy/core/_methods.py:256: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/transitleastsquares/main.py:411: UserWarning: 398 of 402 transits without data. The true period may be twice the given period.
  warnings.warn(text)


TLS Periods with power > 7: [1.89319982 1.8932773  1.89374227]
TIC 213841827, FAP: 0.002641056, SDE: 7.7095963636967815, True Period: 0.53
Detection found: False
❌ Error occurred for TIC 213841827: "['Detection %'] not in index"
TIC 281433945
[0.53] [10]
❌ Error occurred for TIC 281433945: HTTPSConnectionPool(host='mast.stsci.edu', port=443): Read timed out. (read timeout=None)
TIC 33524788
[0.53] [10]
❌ Error occurred for TIC 33524788: HTTPSConnectionPool(host='mast.stsci.edu', port=443): Max retries exceeded with url: /api/v0/invoke (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7feebe7eaaf0>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or not known'))
TIC 78012819
[0.53] [10]


In [ ]:
print(df_results.head(50))
data = df_results[['True Radius (Earth Radii)', 'True Period (Days)', 'Detection %']]
print(data)
pivot = data.pivot(index="True Radius (Earth Radii)", columns="True Period (Days)", values="Detection %")
plt.figure(figsize=(8, 6))
sns.heatmap(pivot, cmap="viridis", vmin=0, vmax=1, cbar_kws={'label': 'Percent found'}).invert_yaxis()
plt.title(f"{tic} noise transit detection rate")
plt.xlabel("Period (Days)")
plt.ylabel("Radius (Earth Radii)")
plt.tight_layout()
plt.savefig(f'heatmap_data/{tic.replace(" ", "_")}_cel_heatmap.png', dpi=300)


os.makedirs("heatmap_data", exist_ok=True)
path = f'heatmap_data/{tic.replace(" ", "_")}_cel_heatmap.csv'
data.to_csv(path, index=False)
print(f"✅ Saved: {path}")
plt.show()


             TIC  True Radius (Earth Radii)  True Period (Days)  \
0  TIC 158235404                        1.0                 1.0   
1   TIC 13145616                        1.0                 1.0   

   Stellar Radius  Stellar Temperature  Stellar Magnitude  Detection  \
0            0.18              31000.0             11.285       True   
1            0.18              29840.0             11.287      False   

   TLS Period  TLS SDE strongest  \
0    1.002834          10.141356   
1    1.991886           5.572499   

                                   TLS SDE > 7 array  \
0  [ 7.35970617  7.04875232  7.81099833  8.919496...   
1                                                 []   

                                   TLS Periods array  TLS Period Uncertainty  \
0  [0.64851778 0.64869227 0.66541326 0.66544936 0...                0.000312   
1                                                 []                0.032420   

   TLS Depth  TLS Rp/Rs  TLS FAP    TLS SNR  Detection %  # tr

ValueError: Index contains duplicate entries, cannot reshape